<a href="https://colab.research.google.com/github/Adityapandeya/Algorithm/blob/main/VYB_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install fuzzywuzzy

In [10]:
import json
from typing import Dict, List, Tuple
# from fuzzywuzzy import fuzz  # For basic fuzzy matching # Removed from here
try:
    from fuzzywuzzy import fuzz  # For basic fuzzy matching
except ImportError:
    print("Error: The 'fuzzywuzzy' module is not installed.")
    print("Please install it using: pip install fuzzywuzzy")
    fuzz = None # Set fuzz to None so the program doesn't crash, and can handle the error.

# Sample Nutrition Database (per 100g)
nutrition_db = {
    "paneer": {"calories": 260, "protein": 18, "carbs": 4, "fat": 21, "fiber": 0},
    "butter": {"calories": 717, "protein": 1, "carbs": 0, "fat": 81, "fiber": 0},
    "tomato": {"calories": 18, "protein": 1, "carbs": 4, "fat": 0, "fiber": 1.5},
    "onion": {"calories": 40, "protein": 1, "carbs": 9, "fat": 0.1, "fiber": 1.7},
    "cream": {"calories": 207, "protein": 3, "carbs": 3, "fat": 21, "fiber": 0},
    "cumin seeds": {"calories": 375, "protein": 18, "carbs": 44, "fat": 22, "fiber": 11},
    "turmeric powder": {"calories": 312, "protein": 10, "carbs": 67, "fat": 3, "fiber": 21},
    "chickpeas": {"calories": 364, "protein": 19, "carbs": 61, "fat": 6, "fiber": 17},
    "spinach": {"calories": 23, "protein": 3, "carbs": 4, "fat": 0.4, "fiber": 2},
    "mustard oil": {"calories": 884, "protein": 0, "carbs": 0, "fat": 100, "fiber": 0},
    "wheat flour": {"calories": 340, "protein": 11, "carbs": 72, "fat": 1.5, "fiber": 12},
    "rice": {"calories": 130, "protein": 3, "carbs": 28, "fat": 0.3, "fiber": 0.4},
    "lentils": {"calories": 116, "protein": 9, "carbs": 20, "fat": 0.4, "fiber": 8},
    "chicken": {"calories": 165, "protein": 31, "carbs": 0, "fat": 3.6, "fiber": 0},
    "ginger": {"calories": 80, "protein": 2, "carbs": 18, "fat": 0.8, "fiber": 2},
    "garlic": {"calories": 149, "protein": 6, "carbs": 33, "fat": 0.5, "fiber": 2.1},
    "coriander leaves": {"calories": 23, "protein": 2, "carbs": 4, "fat": 0.5, "fiber": 2},
}

# Standardized household measurement mapping (example)
household_measurements = {
    "Wet Sabzi": {"katori": {"volume_ml": 200, "approx_weight_g": 180}},
    "Dry Sabzi": {"katori": {"volume_ml": 150, "approx_weight_g": 120}},
    "Dal": {"katori": {"volume_ml": 200, "approx_weight_g": 200}},
    "Non-Veg Curry": {"katori": {"volume_ml": 200, "approx_weight_g": 190}},
    "cup": {"approx_weight_g": 150},  # Generic cup for dry ingredients
    "teaspoon": {"approx_weight_g": 5},
    "tablespoon": {"approx_weight_g": 15},
    "glass": {"volume_ml": 250, "approx_weight_g": 250}, # Assuming liquid
}

# Mapping of food types to standard serving measurements
food_type_serving = {
    "Wet Sabzi": {"unit": "katori", "quantity": 1, "serving_weight_g": 180},
    "Dry Sabzi": {"unit": "katori", "quantity": 1, "serving_weight_g": 120},
    "Dal": {"unit": "katori", "quantity": 1, "serving_weight_g": 200},
    "Non-Veg Curry": {"unit": "katori", "quantity": 1, "serving_weight_g": 190},
    "Roti": {"unit": "number", "quantity": 2, "serving_weight_g": 60}, # Example for other types
    "Rice": {"unit": "katori", "quantity": 1, "serving_weight_g": 150} # Example
}

# Mapping of dish names to food types (very basic)
dish_type_mapping = {
    "Paneer Butter Masala": "Wet Sabzi",
    "Aloo Gobi": "Dry Sabzi",
    "Dal Tadka": "Dal",
    "Butter Chicken": "Non-Veg Curry",
    "Rajma Masala": "Wet Sabzi",
    "Palak Paneer": "Wet Sabzi",
    "Chole Bhature": "Wet Sabzi", # Assuming 'Chole' part
    "Chicken Curry": "Non-Veg Curry",
    "Masoor Dal": "Dal",
    "Bhindi Masala": "Dry Sabzi",
    "Roti": "Roti",
    "Rice": "Rice"
}

# Simulate fetching a generic recipe (ingredient list with approximate quantities)
def fetch_recipe(dish_name: str) -> Dict[str, str]:
    recipes = {
        "Paneer Butter Masala": {
            "paneer": "250g cubes",
            "butter": "2 tbsp",
            "tomato": "1 cup puree",
            "onion": "1 medium chopped",
            "cream": "2 tbsp",
            "ginger-garlic paste": "1 tbsp",
            "cumin seeds": "1 tsp",
            "turmeric powder": "0.5 tsp",
        },
        "Aloo Gobi": {
            "potato": "2 medium",
            "cauliflower": "1 small head",
            "onion": "1 medium chopped",
            "tomato": "1 small chopped",
            "mustard oil": "2 tbsp",
            "turmeric powder": "0.5 tsp",
            "coriander powder": "1 tsp",
        },
        "Dal Tadka": {
            "lentils": "1 cup",
            "onion": "0.5 medium chopped",
            "tomato": "0.5 medium chopped",
            "ghee": "1 tbsp",
            "cumin seeds": "0.5 tsp",
            "turmeric powder": "a pinch",
        },
        "Butter Chicken": {
            "chicken": "500g",
            "butter": "3 tbsp",
            "tomato": "1.5 cup puree",
            "onion": "1 medium paste",
            "cream": "3 tbsp",
            "ginger-garlic paste": "1 tbsp",
            "cashew nuts": "1/4 cup",
        },
        "Rajma Masala": {
            "kidney beans": "1 cup soaked",
            "onion": "1 medium chopped",
            "tomato": "1 cup chopped",
            "ginger-garlic paste": "1 tbsp",
            "cumin powder": "1 tsp",
            "coriander powder": "1 tsp",
        },
        "Roti": {
            "wheat flour": "2 cups",
            "water": "1 cup",
        },
        "Rice":{
            "rice": "1 cup",
            "water": "2 cups"
        }
    }
    return recipes.get(dish_name.title(), {}) # Title case for robustness

# Function to standardize ingredient quantities into household measurements
def standardize_quantity(ingredient: str, quantity_str: str) -> str:
    # This is a very basic attempt. A real-world scenario would need more sophisticated NLP.
    quantity_str = quantity_str.lower()
    if "cup" in quantity_str:
        return quantity_str.replace("cup", "cup")
    elif "tbsp" in quantity_str or "tablespoon" in quantity_str:
        return quantity_str.replace("tbsp", "tablespoons").replace("tablespoon", "tablespoons")
    elif "tsp" in quantity_str or "teaspoon" in quantity_str:
        return quantity_str.replace("tsp", "teaspoons").replace("teaspoon", "teaspoons")
    elif "katori" in quantity_str:
        return quantity_str
    elif "glass" in quantity_str:
        return quantity_str
    elif "medium" in quantity_str or "small" in quantity_str or "large" in quantity_str or "g" in quantity_str:
        # For now, we'll just keep these as is and handle conversion later
        return quantity_str
    elif "pinch" in quantity_str:
        return "a pinch"
    else:
        return quantity_str # Default to original if no clear unit

# Function to map ingredient name to the nutrition database (with basic fuzzy matching)
def map_ingredient_to_nutrition(ingredient_name: str) -> Tuple[str, Dict]:
    best_match = None
    best_ratio = 0
    for db_name in nutrition_db:
        if fuzz: # Check if fuzz is available
            ratio = fuzz.ratio(ingredient_name.lower(), db_name.lower())
            if ratio > best_ratio and ratio > 80: # Threshold for a good match
                best_ratio = ratio
                best_match = db_name
        elif ingredient_name.lower() in db_name.lower() or db_name.lower() in ingredient_name.lower():
            return db_name, nutrition_db[db_name] # Prioritize substring matches

    if best_match:
        return best_match, nutrition_db[best_match]
    else:
        return ingredient_name, {} # Return original name and empty dict if not found

# Function to estimate gram weight from household measurements (very basic)
def estimate_gram_weight(quantity_str: str, ingredient_name: str, food_type: str = "Generic") -> float:
    quantity_str = quantity_str.lower()
    try:
        parts = quantity_str.split()
        value = 0
        unit = ""

        if len(parts) > 0:
            try:
                value = float(parts[0])
            except ValueError:
                value = 0
            unit = " ".join(parts[1:])

        if "cup" in unit:
            return value * household_measurements.get("cup", {}).get("approx_weight_g", 150)
        elif "tablespoon" in unit:
            return value * household_measurements.get("tablespoon", {}).get("approx_weight_g", 15)
        elif "teaspoon" in unit:
            return value * household_measurements.get("teaspoon", {}).get("approx_weight_g", 5)
        elif "katori" in unit and food_type in household_measurements:
            return value * household_measurements[food_type]["katori"]["approx_weight_g"]
        elif "glass" in unit and "liquid" in food_type.lower(): # Very basic liquid assumption
            return value * household_measurements.get("glass", {}).get("approx_weight_g", 250)
        elif "g" in unit:
            return value
        elif "medium" in unit:
            # Very rough estimate for common vegetables
            if "onion" in ingredient_name.lower() or "tomato" in ingredient_name.lower() or "potato" in ingredient_name.lower():
                return value * 150 if value > 0 else 150
        elif "small" in unit:
             if "onion" in ingredient_name.lower() or "tomato" in ingredient_name.lower() or "potato" in ingredient_name.lower():
                return value * 100 if value > 0 else 100
        elif "large" in unit:
            if "onion" in ingredient_name.lower() or "tomato" in ingredient_name.lower() or "potato" in ingredient_name.lower():
                return value * 200 if value > 0 else 200
        elif "head" in unit and "cauliflower" in ingredient_name.lower():
            return value * 500 if value > 0 else 500# Rough estimate
        elif "soaked" in unit and "cup" in unit and "beans" in ingredient_name.lower():
            return value * 200 if value > 0 else 200 # Soaked beans are heavier
        elif "pinch" in unit:
            return 1  # Assume a pinch is 1 gram
        else:
            print(f"Warning: Could not estimate weight for '{quantity_str}' of {ingredient_name}. Defaulting to 100g.")
            return value * 100 if value > 0 else 100 # Default if no conversion found
    except ValueError:
        print(f"Warning: Invalid quantity format '{quantity_str}' for {ingredient_name}. Skipping.")
        return 0

# Function to calculate total nutrition for the dish
def calculate_total_nutrition(ingredients: Dict[str, str], food_type: str) -> Tuple[Dict, List[Dict]]:
    total_nutrition = {"calories": 0, "protein": 0, "carbs": 0, "fat": 0, "fiber": 0}
    ingredients_used = []
    for ingredient_name, quantity_str in ingredients.items():
        mapped_name, nutrition_info = map_ingredient_to_nutrition(ingredient_name)
        standardized_quantity = standardize_quantity(ingredient_name, quantity_str)
        estimated_weight_g = estimate_gram_weight(quantity_str, mapped_name, food_type)

        if nutrition_info:
            for nutrient, value_per_100g in nutrition_info.items():
                total_nutrition[nutrient] += (estimated_weight_g / 100) * value_per_100g
            ingredients_used.append({"ingredient": mapped_name, "quantity": standardized_quantity})
        else:
            print(f"Warning: '{ingredient_name}' not found in nutrition database.")
            ingredients_used.append({"ingredient": ingredient_name, "quantity": standardized_quantity + " (not found in DB)"})

    return total_nutrition, ingredients_used

# Function to identify food type
def identify_food_type(dish_name: str) -> str:
    return dish_type_mapping.get(dish_name.title(), "Unknown")

# Function to extrapolate nutrition for a standard serving
def calculate_nutrition_per_serving(total_nutrition: Dict, food_type: str) -> Dict:
    serving_info = food_type_serving.get(food_type)
    if serving_info and serving_info.get("serving_weight_g"):
        serving_weight_g = serving_info["serving_weight_g"]
        # Assuming the total weight of the recipe is roughly 800g for 3-4 people, unless it is Roti or Rice
        total_weight_g = 800.0
        if food_type == "Roti":
            total_weight_g = 60 * 4 # Assuming 4 rotis
        elif food_type == "Rice":
            total_weight_g = 150 * 4  # Assuming 4 servings of rice
        scaling_factor = serving_weight_g / total_weight_g
        nutrition_per_serving = {nutrient: value * scaling_factor for nutrient, value in total_nutrition.items()}
        return nutrition_per_serving
    else:
        print(f"Warning: Standard serving information not found for '{food_type}'.")
        return {}

# Main function to process a dish name
def estimate_nutrition(dish_name: str) -> Dict:
    recipe = fetch_recipe(dish_name)
    if not recipe:
        return {"error": f"Recipe not found for '{dish_name}'"}

    food_type = identify_food_type(dish_name)

    total_nutrition, ingredients_used = calculate_total_nutrition(recipe, food_type)
    nutrition_per_serving = calculate_nutrition_per_serving(total_nutrition, food_type)
    result = {
        "estimated_nutrition_per_serving": nutrition_per_serving,
        "dish_type": food_type,
        "ingredients_used": ingredients_used,
    }
    return result

# --- Test the function with an example ---
if __name__ == "__main__":
    dish_name = "Paneer Butter Masala"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))

    dish_name = "Aloo Gobi"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))

    dish_name = "Dal Tadka"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))

    dish_name = "Butter Chicken"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))

    dish_name = "Rajma Masala"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))

    dish_name = "Roti"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))

    dish_name = "Rice"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))

    dish_name = "Unknown Dish"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))


{
  "estimated_nutrition_per_serving": {
    "calories": 613.35,
    "protein": 11.700000000000001,
    "carbs": 24.075,
    "fat": 55.94625,
    "fiber": 5.9175
  },
  "dish_type": "Wet Sabzi",
  "ingredients_used": [
    {
      "ingredient": "paneer",
      "quantity": "250g cubes"
    },
    {
      "ingredient": "butter",
      "quantity": "2 tablespoonss"
    },
    {
      "ingredient": "tomato",
      "quantity": "1 cup puree"
    },
    {
      "ingredient": "onion",
      "quantity": "1 medium chopped"
    },
    {
      "ingredient": "cream",
      "quantity": "2 tablespoonss"
    },
    {
      "ingredient": "ginger-garlic paste",
      "quantity": "1 tablespoonss (not found in DB)"
    },
    {
      "ingredient": "cumin seeds",
      "quantity": "1 teaspoonss"
    },
    {
      "ingredient": "turmeric powder",
      "quantity": "0.5 teaspoonss"
    }
  ]
}
{
  "estimated_nutrition_per_serving": {
    "calories": 300.3,
    "protein": 1.125,
    "carbs": 7.649999999999999

In [11]:
import json
from typing import Dict, List, Tuple
# from fuzzywuzzy import fuzz  # For basic fuzzy matching # Removed from here
try:
    from fuzzywuzzy import fuzz  # For basic fuzzy matching
except ImportError:
    print("Error: The 'fuzzywuzzy' module is not installed.")
    print("Please install it using: pip install fuzzywuzzy")
    fuzz = None # Set fuzz to None so the program doesn't crash, and can handle the error.

# Sample Nutrition Database (per 100g)
nutrition_db = {
    "paneer": {"calories": 260, "protein": 18, "carbs": 4, "fat": 21, "fiber": 0},
    "butter": {"calories": 717, "protein": 1, "carbs": 0, "fat": 81, "fiber": 0},
    "tomato": {"calories": 18, "protein": 1, "carbs": 4, "fat": 0, "fiber": 1.5},
    "onion": {"calories": 40, "protein": 1, "carbs": 9, "fat": 0.1, "fiber": 1.7},
    "cream": {"calories": 207, "protein": 3, "carbs": 3, "fat": 21, "fiber": 0},
    "cumin seeds": {"calories": 375, "protein": 18, "carbs": 44, "fat": 22, "fiber": 11},
    "turmeric powder": {"calories": 312, "protein": 10, "carbs": 67, "fat": 3, "fiber": 21},
    "chickpeas": {"calories": 364, "protein": 19, "carbs": 61, "fat": 6, "fiber": 17},
    "spinach": {"calories": 23, "protein": 3, "carbs": 4, "fat": 0.4, "fiber": 2},
    "mustard oil": {"calories": 884, "protein": 0, "carbs": 0, "fat": 100, "fiber": 0},
    "wheat flour": {"calories": 340, "protein": 11, "carbs": 72, "fat": 1.5, "fiber": 12},
    "rice": {"calories": 130, "protein": 3, "carbs": 28, "fat": 0.3, "fiber": 0.4},
    "lentils": {"calories": 116, "protein": 9, "carbs": 20, "fat": 0.4, "fiber": 8},
    "chicken": {"calories": 165, "protein": 31, "carbs": 0, "fat": 3.6, "fiber": 0},
    "ginger": {"calories": 80, "protein": 2, "carbs": 18, "fat": 0.8, "fiber": 2},
    "garlic": {"calories": 149, "protein": 6, "carbs": 33, "fat": 0.5, "fiber": 2.1},
    "coriander leaves": {"calories": 23, "protein": 2, "carbs": 4, "fat": 0.5, "fiber": 2},
}

# Standardized household measurement mapping (example)
household_measurements = {
    "Wet Sabzi": {"katori": {"volume_ml": 200, "approx_weight_g": 180}},
    "Dry Sabzi": {"katori": {"volume_ml": 150, "approx_weight_g": 120}},
    "Dal": {"katori": {"volume_ml": 200, "approx_weight_g": 200}},
    "Non-Veg Curry": {"katori": {"volume_ml": 200, "approx_weight_g": 190}},
    "cup": {"approx_weight_g": 150},  # Generic cup for dry ingredients
    "teaspoon": {"approx_weight_g": 5},
    "tablespoon": {"approx_weight_g": 15},
    "glass": {"volume_ml": 250, "approx_weight_g": 250}, # Assuming liquid
}

# Mapping of food types to standard serving measurements
food_type_serving = {
    "Wet Sabzi": {"unit": "katori", "quantity": 1, "serving_weight_g": 180},
    "Dry Sabzi": {"unit": "katori", "quantity": 1, "serving_weight_g": 120},
    "Dal": {"unit": "katori", "quantity": 1, "serving_weight_g": 200},
    "Non-Veg Curry": {"unit": "katori", "quantity": 1, "serving_weight_g": 190},
    "Roti": {"unit": "number", "quantity": 2, "serving_weight_g": 60}, # Example for other types
    "Rice": {"unit": "katori", "quantity": 1, "serving_weight_g": 150} # Example
}

# Mapping of dish names to food types (very basic)
dish_type_mapping = {
    "Paneer Butter Masala": "Wet Sabzi",
    "Aloo Gobi": "Dry Sabzi",
    "Dal Tadka": "Dal",
    "Butter Chicken": "Non-Veg Curry",
    "Rajma Masala": "Wet Sabzi",
    "Palak Paneer": "Wet Sabzi",
    "Chole Bhature": "Wet Sabzi", # Assuming 'Chole' part
    "Chicken Curry": "Non-Veg Curry",
    "Masoor Dal": "Dal",
    "Bhindi Masala": "Dry Sabzi",
    "Roti": "Roti",
    "Rice": "Rice"
}

# Simulate fetching a generic recipe (ingredient list with approximate quantities)
def fetch_recipe(dish_name: str) -> Dict[str, str]:
    recipes = {
        "Paneer Butter Masala": {
            "paneer": "250g cubes",
            "butter": "2 tbsp",
            "tomato": "1 cup puree",
            "onion": "1 medium chopped",
            "cream": "2 tbsp",
            "ginger-garlic paste": "1 tbsp",
            "cumin seeds": "1 tsp",
            "turmeric powder": "0.5 tsp",
        },
        "Aloo Gobi": {
            "potato": "2 medium",
            "cauliflower": "1 small head",
            "onion": "1 medium chopped",
            "tomato": "1 small chopped",
            "mustard oil": "2 tbsp",
            "turmeric powder": "0.5 tsp",
            "coriander powder": "1 tsp",
        },
        "Dal Tadka": {
            "lentils": "1 cup",
            "onion": "0.5 medium chopped",
            "tomato": "0.5 medium chopped",
            "ghee": "1 tbsp",
            "cumin seeds": "0.5 tsp",
            "turmeric powder": "a pinch",
        },
        "Butter Chicken": {
            "chicken": "500g",
            "butter": "3 tbsp",
            "tomato": "1.5 cup puree",
            "onion": "1 medium paste",
            "cream": "3 tbsp",
            "ginger-garlic paste": "1 tbsp",
            "cashew nuts": "1/4 cup",
        },
        "Rajma Masala": {
            "kidney beans": "1 cup soaked",
            "onion": "1 medium chopped",
            "tomato": "1 cup chopped",
            "ginger-garlic paste": "1 tbsp",
            "cumin powder": "1 tsp",
            "coriander powder": "1 tsp",
        },
        "Roti": {
            "wheat flour": "2 cups",
            "water": "1 cup",
        },
        "Rice":{
            "rice": "1 cup",
            "water": "2 cups"
        }
    }
    return recipes.get(dish_name.title(), {}) # Title case for robustness

# Function to standardize ingredient quantities into household measurements
def standardize_quantity(ingredient: str, quantity_str: str) -> str:
    # This is a very basic attempt. A real-world scenario would need more sophisticated NLP.
    quantity_str = quantity_str.lower()
    if "cup" in quantity_str:
        return quantity_str.replace("cup", " cup")
    elif "tbsp" in quantity_str or "tablespoon" in quantity_str:
        return quantity_str.replace("tbsp", " tablespoons").replace("tablespoon", " tablespoons")
    elif "tsp" in quantity_str or "teaspoon" in quantity_str:
        return quantity_str.replace("tsp", " teaspoons").replace("teaspoon", " teaspoons")
    elif "katori" in quantity_str:
        return quantity_str.replace("katori", " katori")
    elif "glass" in quantity_str:
        return quantity_str.replace("glass", " glass")
    elif "medium" in quantity_str or "small" in quantity_str or "large" in quantity_str or "g" in quantity_str:
        # For now, we'll just keep these as is and handle conversion later
        return quantity_str
    elif "pinch" in quantity_str:
        return "a pinch"
    else:
        return quantity_str # Default to original if no clear unit

# Function to map ingredient name to the nutrition database (with basic fuzzy matching)
def map_ingredient_to_nutrition(ingredient_name: str) -> Tuple[str, Dict]:
    best_match = None
    best_ratio = 0
    for db_name in nutrition_db:
        if fuzz: # Check if fuzz is available
            ratio = fuzz.ratio(ingredient_name.lower(), db_name.lower())
            if ratio > best_ratio and ratio > 80: # Threshold for a good match
                best_ratio = ratio
                best_match = db_name
        elif ingredient_name.lower() in db_name.lower() or db_name.lower() in ingredient_name.lower():
            return db_name, nutrition_db[db_name] # Prioritize substring matches

    if best_match:
        return best_match, nutrition_db[best_match]
    else:
        return ingredient_name, {} # Return original name and empty dict if not found

# Function to estimate gram weight from household measurements (very basic)
def estimate_gram_weight(quantity_str: str, ingredient_name: str, food_type: str = "Generic") -> float:
    quantity_str = quantity_str.lower()
    try:
        parts = quantity_str.split()
        value = 0
        unit = ""

        if len(parts) > 0:
            try:
                value = float(parts[0])
            except ValueError:
                value = 0
            unit = " ".join(parts[1:])

        if "cup" in unit:
            return value * household_measurements.get("cup", {}).get("approx_weight_g", 150)
        elif "tablespoon" in unit:
            return value * household_measurements.get("tablespoon", {}).get("approx_weight_g", 15)
        elif "teaspoon" in unit:
            return value * household_measurements.get("teaspoon", {}).get("approx_weight_g", 5)
        elif "katori" in unit and food_type in household_measurements:
            return value * household_measurements[food_type]["katori"]["approx_weight_g"]
        elif "glass" in unit and "liquid" in food_type.lower(): # Very basic liquid assumption
            return value * household_measurements.get("glass", {}).get("approx_weight_g", 250)
        elif "g" in unit:
            return value
        elif "medium" in unit:
            # Very rough estimate for common vegetables
            if "onion" in ingredient_name.lower() or "tomato" in ingredient_name.lower() or "potato" in ingredient_name.lower():
                return value * 150 if value > 0 else 150
        elif "small" in unit:
             if "onion" in ingredient_name.lower() or "tomato" in ingredient_name.lower() or "potato" in ingredient_name.lower():
                return value * 100 if value > 0 else 100
        elif "large" in unit:
            if "onion" in ingredient_name.lower() or "tomato" in ingredient_name.lower() or "potato" in ingredient_name.lower():
                return value * 200 if value > 0 else 200
        elif "head" in unit and "cauliflower" in ingredient_name.lower():
            return value * 500 if value > 0 else 500# Rough estimate
        elif "soaked" in unit and "cup" in unit and "beans" in ingredient_name.lower():
            return value * 200 if value > 0 else 200 # Soaked beans are heavier
        elif "pinch" in unit:
            return 1  # Assume a pinch is 1 gram
        else:
            print(f"Warning: Could not estimate weight for '{quantity_str}' of {ingredient_name}. Defaulting to 100g.")
            return value * 100 if value > 0 else 100 # Default if no conversion found
    except ValueError:
        print(f"Warning: Invalid quantity format '{quantity_str}' for {ingredient_name}. Skipping.")
        return 0

# Function to calculate total nutrition for the dish
def calculate_total_nutrition(ingredients: Dict[str, str], food_type: str) -> Tuple[Dict, List[Dict]]:
    total_nutrition = {"calories": 0, "protein": 0, "carbs": 0, "fat": 0, "fiber": 0}
    ingredients_used = []
    for ingredient_name, quantity_str in ingredients.items():
        mapped_name, nutrition_info = map_ingredient_to_nutrition(ingredient_name)
        standardized_quantity = standardize_quantity(ingredient_name, quantity_str)
        estimated_weight_g = estimate_gram_weight(quantity_str, mapped_name, food_type)

        if nutrition_info:
            for nutrient, value_per_100g in nutrition_info.items():
                total_nutrition[nutrient] += (estimated_weight_g / 100) * value_per_100g
            ingredients_used.append({"ingredient": mapped_name, "quantity": standardized_quantity})
        else:
            print(f"Warning: '{ingredient_name}' not found in nutrition database.")
            ingredients_used.append({"ingredient": ingredient_name, "quantity": standardized_quantity + " (not found in DB)"})

    return total_nutrition, ingredients_used

# Function to identify food type
def identify_food_type(dish_name: str) -> str:
    return dish_type_mapping.get(dish_name.title(), "Unknown")

# Function to extrapolate nutrition for a standard serving
def calculate_nutrition_per_serving(total_nutrition: Dict, food_type: str) -> Dict:
    serving_info = food_type_serving.get(food_type)
    if serving_info and serving_info.get("serving_weight_g"):
        serving_weight_g = serving_info["serving_weight_g"]
        # Assuming the total weight of the recipe is roughly 800g for 3-4 people, unless it is Roti or Rice
        total_weight_g = 800.0
        if food_type == "Roti":
            total_weight_g = 60 * 4 # Assuming 4 rotis
        elif food_type == "Rice":
            total_weight_g = 150 * 4  # Assuming 4 servings of rice
        scaling_factor = serving_weight_g / total_weight_g
        nutrition_per_serving = {nutrient: value * scaling_factor for nutrient, value in total_nutrition.items()}
        return nutrition_per_serving
    else:
        print(f"Warning: Standard serving information not found for '{food_type}'.")
        return {}

# Main function to process a dish name
def estimate_nutrition(dish_name: str) -> Dict:
    recipe = fetch_recipe(dish_name)
    if not recipe:
        return {"error": f"Recipe not found for '{dish_name}'"}

    food_type = identify_food_type(dish_name)

    total_nutrition, ingredients_used = calculate_total_nutrition(recipe, food_type)
    nutrition_per_serving = calculate_nutrition_per_serving(total_nutrition, food_type)
    result = {
        "estimated_nutrition_per_serving": nutrition_per_serving,
        "dish_type": food_type,
        "ingredients_used": ingredients_used,
    }
    return result

# --- Test the function with an example ---
if __name__ == "__main__":
    dish_name = "Paneer Butter Masala"
    result = estimate_nutrition(dish_name)
    print(json.dumps(result, indent=2))


{
  "estimated_nutrition_per_serving": {
    "calories": 613.35,
    "protein": 11.700000000000001,
    "carbs": 24.075,
    "fat": 55.94625,
    "fiber": 5.9175
  },
  "dish_type": "Wet Sabzi",
  "ingredients_used": [
    {
      "ingredient": "paneer",
      "quantity": "250g cubes"
    },
    {
      "ingredient": "butter",
      "quantity": "2   tablespoonss"
    },
    {
      "ingredient": "tomato",
      "quantity": "1  cup puree"
    },
    {
      "ingredient": "onion",
      "quantity": "1 medium chopped"
    },
    {
      "ingredient": "cream",
      "quantity": "2   tablespoonss"
    },
    {
      "ingredient": "ginger-garlic paste",
      "quantity": "1   tablespoonss (not found in DB)"
    },
    {
      "ingredient": "cumin seeds",
      "quantity": "1   teaspoonss"
    },
    {
      "ingredient": "turmeric powder",
      "quantity": "0.5   teaspoonss"
    }
  ]
}
